# 11 — Exploitation du signal cross-sectionnel

## Objectif

Évaluer si la position relative d’une allocation au sein d’une même date apporte une information prédictive complémentaire aux rendements historiques.

Nous comparons deux modèles Gradient Boosting :

- **Modèle de référence** : `RET_1` à `RET_20`
- **Modèle challenger** : `RET_1` à `RET_20` + `RET_1_percentile_by_date`

## Feature cross-sectionnelle étudiée

La variable `RET_1_percentile_by_date` représente le rang percentile de `RET_1` parmi toutes les allocations observées à la même date.

Pour une allocation $i$ observée à la date $t$ :

$$
Q_{i,t}
=
\operatorname{PercentileRank}
\left(
RET_{1,i,t}
\mid
\left\{RET_{1,j,t}\right\}_{j=1}^{N_t}
\right)
$$

où $N_t$ représente le nombre d’allocations disponibles à la date $t$.

Une valeur proche de 1 indique que l’allocation possède l’un des rendements récents les plus élevés de sa date. Une valeur percentile faible indique une position relative faible.

## Hypothèse de recherche

Les rendements historiques décrivent chaque allocation en valeur absolue, mais ne représentent pas directement sa position relative parmi les autres allocations de la même date.

Nous testons si l’ajout de `RET_1_percentile_by_date` permet au Gradient Boosting de mieux distinguer et classer les allocations dont la performance future sera positive.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

In [2]:
import numpy as np
import pandas as pd
from src.features import ret_features
from src.data_loading import load_X_train, load_y_train
from src.validation import create_expanding_window_folds, check_temporal_folds
from src.evaluation import evaluate_model_on_folds, compare_model_results
from src.boosting_models import build_gradboosting_pipeline
from src.target import create_class_column
from src.cross_sectional_features import create_return_percentile_by_date

In [3]:
X_train = load_X_train()
y_train = load_y_train()

In [4]:
df_train, _  = create_class_column(X_train, y_train)

In [5]:
dates = sorted(list(set(X_train["TS"])))
folds = create_expanding_window_folds(dates)
check_temporal_folds(folds,dates,validation_size=120)

True

In [6]:
df_train_with_percentile = create_return_percentile_by_date(df_train)

In [7]:
df_train_with_percentile.columns

Index(['ROW_ID', 'TS', 'ALLOCATION', 'RET_20', 'RET_19', 'RET_18', 'RET_17',
       'RET_16', 'RET_15', 'RET_14', 'RET_13', 'RET_12', 'RET_11', 'RET_10',
       'RET_9', 'RET_8', 'RET_7', 'RET_6', 'RET_5', 'RET_4', 'RET_3', 'RET_2',
       'RET_1', 'SIGNED_VOLUME_20', 'SIGNED_VOLUME_19', 'SIGNED_VOLUME_18',
       'SIGNED_VOLUME_17', 'SIGNED_VOLUME_16', 'SIGNED_VOLUME_15',
       'SIGNED_VOLUME_14', 'SIGNED_VOLUME_13', 'SIGNED_VOLUME_12',
       'SIGNED_VOLUME_11', 'SIGNED_VOLUME_10', 'SIGNED_VOLUME_9',
       'SIGNED_VOLUME_8', 'SIGNED_VOLUME_7', 'SIGNED_VOLUME_6',
       'SIGNED_VOLUME_5', 'SIGNED_VOLUME_4', 'SIGNED_VOLUME_3',
       'SIGNED_VOLUME_2', 'SIGNED_VOLUME_1', 'MEDIAN_DAILY_TURNOVER', 'GROUP',
       'target', 'class', 'RET_1_percentile_by_date'],
      dtype='str')

In [8]:
percentile_column = "RET_1_percentile_by_date"

assert len(df_train) == len(df_train_with_percentile)

assert df_train["ROW_ID"].equals(
    df_train_with_percentile["ROW_ID"]
)

assert df_train.index.equals(
    df_train_with_percentile.index
)

assert percentile_column in df_train_with_percentile.columns

non_missing_percentiles = (
    df_train_with_percentile[percentile_column]
    .dropna()
)

assert non_missing_percentiles.gt(0.0).all()
assert non_missing_percentiles.le(1.0).all()

ret1_missing_mask = (
    df_train_with_percentile["RET_1"].isna()
)

percentile_missing_mask = (
    df_train_with_percentile[percentile_column].isna()
)

assert ret1_missing_mask.equals(
    percentile_missing_mask
)

In [9]:
ret_columns = ret_features(df_train)

ret_columns_with_percentile = (
    ret_columns
    + ["RET_1_percentile_by_date"]
)

In [10]:
assert len(ret_columns) == 20
assert len(ret_columns_with_percentile) == 21

assert len(set(ret_columns_with_percentile)) == len(
    ret_columns_with_percentile
)

In [11]:
gradient_boosting_reference = build_gradboosting_pipeline()
gradient_boosting_percentile = build_gradboosting_pipeline()

In [12]:
gradient_boosting_results = evaluate_model_on_folds(
    df_train,
    folds,
    gradient_boosting_reference,
    ret_columns
)

gradient_boosting_percentile_results = evaluate_model_on_folds(
    df_train_with_percentile,
    folds,
    gradient_boosting_percentile,
    ret_columns_with_percentile
)

In [13]:
comparison_results = compare_model_results(
    results_a=gradient_boosting_results,
    results_b=gradient_boosting_percentile_results,
    model_a_name="reference",
    model_b_name="percentile",
)

comparison_results

,fold,accuracy_reference,log_loss_reference,roc_auc_reference,accuracy_percentile,log_loss_percentile,roc_auc_percentile,delta_accuracy,delta_log_loss,delta_roc_auc
0,1,0.519283,0.692187,0.519829,0.518578,0.692224,0.518304,-0.000705,0.000036,-0.001524
1,2,0.533694,0.691233,0.549852,0.536317,0.690929,0.550817,0.002623,-0.000304,0.000965
2,3,0.524108,0.692039,0.523544,0.522008,0.691912,0.524621,-0.002100,-0.000127,0.001077
3,4,0.523889,0.691922,0.524227,0.520655,0.691885,0.523070,-0.003235,-0.000037,-0.001157
